# Olympic Data Preprocessing

This notebook takes the raw Olympic athlete records and turns them into something our D3 dashboard can actually use. Along the way we clean up missing values, engineer a few time-based features (decade, era), map every country to a continent, and finally export four lightweight JSON files that get loaded straight into the browser.

We also keep a small, manually curated table of Olympic disruptions — boycotts, cancelled Games, wars — because that information simply isn't present in either of our two source datasets and has to be added as historical context rather than computed.

**What comes out at the end:**
- `python/olympics_master_cleaned.csv` — the full cleaned athlete table (271K rows)
- `python/world_events_clean.csv` — merged geopolitical events (~440 rows)
- `template/public/data/viz1_medal_share.json` — continent medal share by year
- `template/public/data/viz2_efficiency.json` — medal efficiency by country/decade
- `template/public/data/viz3_network.json` — country–sport dominance network by era
- `template/public/data/olympic_disruptions.json` — boycotts, cancellations, wars, terrorism


## 1. Setup — Paths & Imports

Before touching any data, we point the notebook at the right folders. We use **relative paths** instead of hardcoded absolute paths — that way the notebook runs correctly no matter whose machine it's on, without anyone having to manually edit file paths.

In [1]:
import pandas as pd
import numpy as np
import os

# This notebook lives inside assignment_05/python/, so everything is relative to that.
LAB = os.path.abspath('.')                                              # python/  (raw CSVs + this notebook)
PYTHON_OUT = LAB                                                        # cleaned CSVs also saved here
DATA_OUT = os.path.abspath(os.path.join(LAB, '..', 'template', 'public', 'data'))  # JSONs for the browser

os.makedirs(PYTHON_OUT, exist_ok=True)
os.makedirs(DATA_OUT, exist_ok=True)

print('✅ Paths set')
print(f'  Source (python/)      : {LAB}')
print(f'  Cleaned CSVs → python/: {PYTHON_OUT}')
print(f'  JSONs → public/data/  : {DATA_OUT}')


✅ Paths set
  Source (python/)      : C:\Users\namit\Desktop\2026-ss-st192675\assignment_05\python
  Cleaned CSVs → python/: C:\Users\namit\Desktop\2026-ss-st192675\assignment_05\python
  JSONs → public/data/  : C:\Users\namit\Desktop\2026-ss-st192675\assignment_05\template\public\data


## 2. Load Raw Data

We start with the two core files: 271K individual athlete-event records, and a small lookup table mapping each NOC (National Olympic Committee) code to a region name. A quick `isnull().sum()` check tells us where the gaps are before we decide how to handle them.

In [2]:
athletes = pd.read_csv(os.path.join(LAB, 'athlete_events.csv'))
noc = pd.read_csv(os.path.join(LAB, 'noc_regions.csv'))

print(f'Athletes: {athletes.shape}')
print(f'NOC: {noc.shape}')
print('\nMissing values:')
print(athletes.isnull().sum())


Athletes: (271116, 15)
NOC: (230, 3)

Missing values:
ID             0
Name           0
Sex            0
Age         9474
Height     60171
Weight     62875
Team           0
NOC            0
Games          0
Year           0
Season         0
City           0
Sport          0
Event          0
Medal     231333
dtype: int64


## 3. Clean Medal Values

A missing `Medal` value doesn't mean bad data — it means the athlete didn't win one. So we fill `NaN` with the string `'None'` and add a simple binary flag, `Medal_Won`, which just makes later groupby aggregations much easier (summing a 0/1 column is cleaner than counting non-null strings).

\( \text{Medal\_Won} = \begin{cases} 1 & \text{if Medal} \neq \text{None} \\ 0 & \text{otherwise} \end{cases} \)

In [3]:
athletes['Medal'] = athletes['Medal'].fillna('None')
athletes['Medal_Won'] = (athletes['Medal'] != 'None').astype(int)
print('Medal distribution:')
print(athletes['Medal'].value_counts())


Medal distribution:
Medal
None      231333
Gold       13372
Bronze     13295
Silver     13116
Name: count, dtype: int64


## 4. Add Time-Based Features

Two new columns make the later visualizations possible:

- **Decade** — we floor each year down to the nearest 10, using \( \text{Decade} = \lfloor \text{Year} / 10 \rfloor \times 10 \). This is what Viz 2 groups by, since looking at 120 individual years would be far too noisy.
- **Era** — a simple three-bucket split (Pre-WWII, Cold War, Post-Cold War) that gives Viz 3's network diagram a manageable number of time slices instead of dozens of individual years.

In [4]:
athletes['Decade'] = (athletes['Year'] // 10) * 10

def get_era(year):
    if year <= 1944: return 'Pre-WWII'
    elif year <= 1988: return 'Cold War'
    else: return 'Post-Cold War'

athletes['Era'] = athletes['Year'].apply(get_era)

print(athletes[['Year','Decade','Era','Medal','Medal_Won']].drop_duplicates('Year').sort_values('Year').head(10))


      Year  Decade       Era   Medal  Medal_Won
3079  1896    1890  Pre-WWII    None          0
3     1900    1900  Pre-WWII    Gold          1
711   1904    1900  Pre-WWII    Gold          1
268   1906    1900  Pre-WWII  Bronze          1
1149  1908    1900  Pre-WWII    None          0
35    1912    1910  Pre-WWII    None          0
2     1920    1920  Pre-WWII    None          0
39    1924    1920  Pre-WWII    None          0
133   1928    1920  Pre-WWII    None          0
26    1932    1930  Pre-WWII    None          0


## 5. Merge NOC Regions & Map Continents

The athlete table only has a 3-letter NOC code (like `USA` or `GER`), not a continent — so we merge in the `region` column from our lookup table, then run every region through a manual continent-assignment function. This is a straightforward dictionary lookup rather than anything statistical, but it has to be done carefully since a few regions (e.g. Kosovo, Individual Olympic Athletes) don't map cleanly onto any single continent.

In [5]:
athletes = athletes.merge(noc[['NOC','region']], on='NOC', how='left')
print('Columns after merge:', athletes.columns.tolist())
print('Sample regions:', athletes['region'].dropna().unique()[:10])


Columns after merge: ['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal', 'Medal_Won', 'Decade', 'Era', 'region']
Sample regions: ['China' 'Denmark' 'Netherlands' 'USA' 'Finland' 'Norway' 'Romania'
 'Estonia' 'France' 'Morocco']


In [6]:
def assign_continent(region):
    europe = {
        'Albania','Andorra','Austria','Belarus','Belgium','Bosnia and Herzegovina',
        'Bulgaria','Croatia','Cyprus','Czech Republic','Denmark','Estonia','Finland',
        'France','Germany','Greece','Hungary','Iceland','Ireland','Italy','Latvia',
        'Liechtenstein','Lithuania','Luxembourg','Malta','Moldova','Monaco',
        'Montenegro','Netherlands','Norway','Poland','Portugal','Romania','Russia',
        'San Marino','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland',
        'Ukraine','United Kingdom','Yugoslavia','Kosovo','North Macedonia',
        'UK','Armenia','Azerbaijan','Georgia','Macedonia'
    }
    americas = {
        'Argentina','Bolivia','Brazil','Canada','Chile','Colombia','Costa Rica',
        'Cuba','Dominican Republic','Ecuador','El Salvador','Guatemala','Haiti',
        'Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru',
        'Puerto Rico','Trinidad','Uruguay','USA','Venezuela','Bahamas','Barbados',
        'Belize','Guyana','Suriname','Trinidad and Tobago',
        'Virgin Islands, US','Virgin Islands, British','American Samoa','Bermuda',
        'Cayman Islands','Aruba','Curacao','Antigua','Grenada','Saint Kitts',
        'Saint Vincent','Saint Lucia','Dominica','Guam','Boliva'
    }
    asia = {
        'Afghanistan','Bahrain','Bangladesh','Bhutan','Brunei','Cambodia','China',
        'India','Indonesia','Iran','Iraq','Israel','Japan','Jordan','Kazakhstan',
        'Kuwait','Kyrgyzstan','Laos','Lebanon','Malaysia','Maldives','Mongolia',
        'Myanmar','Nepal','North Korea','Oman','Pakistan','Palestine','Philippines',
        'Qatar','Saudi Arabia','Singapore','South Korea','Sri Lanka','Syria',
        'Taiwan','Tajikistan','Thailand','Timor-Leste','Turkey','Turkmenistan',
        'UAE','Uzbekistan','Vietnam','Yemen','United Arab Emirates'
    }
    africa = {
        'Algeria','Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon',
        'Cape Verde','Central African Republic','Chad','Comoros','Congo',
        'Djibouti','Egypt','Eritrea','Ethiopia','Gabon','Gambia','Ghana','Guinea',
        'Guinea-Bissau','Ivory Coast','Kenya','Lesotho','Liberia','Libya',
        'Madagascar','Malawi','Mali','Mauritania','Mauritius','Morocco',
        'Mozambique','Namibia','Niger','Nigeria','Rwanda','Senegal',
        'Sierra Leone','Somalia','South Africa','South Sudan','Sudan',
        'Swaziland','Tanzania','Togo','Tunisia','Uganda','Zambia','Zimbabwe',
        'Republic of Congo','Democratic Republic of the Congo','Equatorial Guinea',
        'Seychelles','Sao Tome and Principe'
    }
    oceania = {
        'Australia','Fiji','Kiribati','Marshall Islands','Micronesia','Nauru',
        'New Zealand','Palau','Papua New Guinea','Samoa','Solomon Islands',
        'Tonga','Tuvalu','Vanuatu','Cook Islands'
    }
    if pd.isna(region): return 'Unknown'
    if region in europe: return 'Europe'
    if region in americas: return 'Americas'
    if region in asia: return 'Asia'
    if region in africa: return 'Africa'
    if region in oceania: return 'Oceania'
    if region == 'Individual Olympic Athletes': return 'Unknown'
    return 'Unknown'

athletes['Continent'] = athletes['region'].apply(assign_continent)
print(athletes['Continent'].value_counts())
print('\nUnresolved regions:', athletes[athletes['Continent']=='Unknown']['region'].unique())


Continent
Europe      162620
Americas     51661
Asia         33673
Africa       11992
Oceania      10706
Unknown        464
Name: count, dtype: int64

Unresolved regions: [nan 'Individual Olympic Athletes']


## 6. Build & Save Master CSV

Now that every derived column exists, we trim the table down to just the fields we'll actually need downstream and save it as our "master" cleaned file. This file stays inside `python/` — it's an internal working file, not something the browser ever touches directly.

In [7]:
master = athletes[[
    'ID','Name','Sex','Age','Height','Weight','Team','NOC','region','Continent',
    'Games','Year','Decade','Era','Season','City','Sport','Event','Medal','Medal_Won'
]].copy()

master_path = os.path.join(PYTHON_OUT, 'olympics_master_cleaned.csv')
master.to_csv(master_path, index=False)

print(f'✅ Master CSV saved → {master_path}')
print(f'  Shape: {master.shape}')
print(master.head(3))


✅ Master CSV saved → C:\Users\namit\Desktop\2026-ss-st192675\assignment_05\python\olympics_master_cleaned.csv
  Shape: (271116, 20)
   ID                 Name Sex   Age  Height  Weight     Team  NOC   region  \
0   1            A Dijiang   M  24.0   180.0    80.0    China  CHN    China   
1   2             A Lamusi   M  23.0   170.0    60.0    China  CHN    China   
2   3  Gunnar Nielsen Aaby   M  24.0     NaN     NaN  Denmark  DEN  Denmark   

  Continent        Games  Year  Decade            Era  Season       City  \
0      Asia  1992 Summer  1992    1990  Post-Cold War  Summer  Barcelona   
1      Asia  2012 Summer  2012    2010  Post-Cold War  Summer     London   
2    Europe  1920 Summer  1920    1920       Pre-WWII  Summer  Antwerpen   

        Sport                         Event Medal  Medal_Won  
0  Basketball   Basketball Men's Basketball  None          0  
1        Judo  Judo Men's Extra-Lightweight  None          0  
2    Football       Football Men's Football  None        

## 7. World Events — Load, Clean, Merge & Save

This is our secondary dataset, giving Viz 1's timeline some geopolitical context to sit alongside the medal data. We combine two very different sources:

- **World Important Dates (Kaggle)** — a broad list of global historical incidents, which we filter down to just the categories relevant to our story (wars, political upheaval, policy shifts, civil rights movements, terrorism).
- **UCDP/PRIO Armed Conflict Dataset v26.1** — a more rigorous, academically maintained conflict dataset, which we filter to only the highest-intensity conflicts (full-scale wars, not minor skirmishes) so the timeline isn't cluttered.

We merge both into a single tidy table with the same columns, so the frontend doesn't need to know which source any given row came from.

In [8]:
world_raw = pd.read_csv(os.path.join(LAB, 'World Important Dates.csv'))
ucdp_raw = pd.read_csv(os.path.join(LAB, 'UcdpPrioConflict_v26_1.csv'))
print(f'World Events: {len(world_raw)} rows | UCDP: {len(ucdp_raw)} rows')


World Events: 1096 rows | UCDP: 2816 rows


First we narrow the Kaggle dataset down to the modern Olympic era (1896–2016) and keep only rows whose event type actually matters for our story — we group ~40 raw event-type labels into 5 clean categories (War, Political, Policy, Civil Rights, Terrorism).

In [9]:
# Filter World Important Dates to the Olympic era & relevant categories
world_raw['Year'] = pd.to_numeric(world_raw['Year'], errors='coerce')
world_raw = world_raw.dropna(subset=['Year'])
world_raw['Year'] = world_raw['Year'].astype(int)
world_raw = world_raw[(world_raw['Year'] >= 1896) & (world_raw['Year'] <= 2016)].copy()

war_types = [
    'Military Conflict','Military','Civil War','War','Massacre','Military Occupation',
    'Military Invasion','Military Siege','Military Intervention','Military Engagement',
    'Military Operation','Military Aggression','Military Battle','Military Offensive',
    'Regional Conflict','Ethnic Conflict','Colonial War','Armed Rebellion','Rebellion',
    'Insurgency','Genocide','Military Coup','War Declaration','Military Campaign'
]
political_types = [
    'Political','Political Change','Political Movement','Political Reform',
    'Political Crisis','Political Transition','Revolution','Coup',"Coup d'état",
    'Political Revolution','Political Repression','Political Uprising',
    'Democratic Uprising','Political Scandal','Political Declaration',
    'Political Rebellion','Military Rebellion'
]
policy_types = [
    'Economic Policy','Social Policy','Defense Policy','Health Policy',
    'Environmental Policy','Education Policy','Agricultural Policy',
    'Legislation','Legislative','Legal','Constitutional Development',
    'Constitutional','Constitutional Reform','International Agreement',
    'International Treaty','Treaty','Peace Agreement','Diplomatic Agreement'
]
civil_rights_types = [
    'Civil Rights','Social Justice','Social Movement','Civil Disobedience',
    'Civil Disobedience Movement','Human Rights Improvement','Civil Rights Violation'
]
terrorism_types = ['Terrorism','Domestic Terrorism']

all_relevant = set(war_types + political_types + policy_types + civil_rights_types + terrorism_types)
world_filtered = world_raw[world_raw['Type of Event'].isin(all_relevant)].copy()

def assign_category(event_type):
    if event_type in war_types: return 'War'
    if event_type in terrorism_types: return 'Terrorism'
    if event_type in civil_rights_types: return 'Civil Rights'
    if event_type in policy_types: return 'Policy'
    if event_type in political_types: return 'Political'
    return 'Other'

world_final = pd.DataFrame({
    'Year': world_filtered['Year'],
    'Event': world_filtered['Name of Incident'],
    'Country': world_filtered['Country'],
    'Category': world_filtered['Type of Event'].apply(assign_category),
    'Source': 'World Important Dates (Kaggle)'
})
print(f'World events filtered: {len(world_final)} rows')


World events filtered: 323 rows


Now the UCDP conflict data. We only keep `intensity_level == 2`, which UCDP defines as full-scale war (at least 1,000 battle-related deaths in a year) rather than minor armed conflict — this keeps our timeline focused on genuinely major events. We also deduplicate by `conflict_id` so a long-running war doesn't show up as a dozen near-identical rows.

In [10]:
# Filter UCDP to full-scale wars (intensity_level = 2), deduplicate by conflict_id
ucdp_filtered = ucdp_raw[
    (ucdp_raw['year'] >= 1896) &
    (ucdp_raw['year'] <= 2016) &
    (ucdp_raw['intensity_level'] == 2)
].copy()
ucdp_filtered = ucdp_filtered.sort_values('year')
ucdp_deduped = ucdp_filtered.drop_duplicates(subset=['conflict_id']).copy()

ucdp_final = pd.DataFrame({
    'Year': ucdp_deduped['year'],
    'Event': (ucdp_deduped['location'].astype(str) + ': ' +
              ucdp_deduped['side_a'].astype(str) + ' vs ' +
              ucdp_deduped['side_b'].astype(str)),
    'Country': ucdp_deduped['location'].astype(str),
    'Category': 'War',
    'Source': 'UCDP/PRIO Armed Conflict Dataset v26.1'
})
print(f'UCDP wars deduplicated: {len(ucdp_final)} rows')


UCDP wars deduplicated: 127 rows


In [11]:
# Merge both event datasets into one tidy table
world_events = pd.concat([world_final, ucdp_final], ignore_index=True)
world_events = world_events.sort_values('Year').reset_index(drop=True)
world_events = world_events.drop_duplicates(subset=['Year','Event'])

we_path = os.path.join(PYTHON_OUT, 'world_events_clean.csv')
world_events.to_csv(we_path, index=False)

print(f'✅ World events saved → {we_path}')
print(f'  Shape: {world_events.shape}')
print(world_events['Category'].value_counts())


✅ World events saved → C:\Users\namit\Desktop\2026-ss-st192675\assignment_05\python\world_events_clean.csv
  Shape: (440, 5)
Category
War             219
Political       108
Policy           85
Civil Rights     14
Terrorism        14
Name: count, dtype: int64


## 8. Export Viz JSONs → `template/public/data/`

This is where we hand off to the frontend. Each of the three visualizations needs its data pre-aggregated in Python rather than in the browser — doing it here keeps the JSON files small and keeps D3 focused purely on rendering, not on crunching 271K rows client-side.

**Viz 1 — Athlete Participation by Continent.** For every Summer Olympic year, we count unique athletes per continent and turn that into a percentage of that year's total unique-athlete count:

$$ \text{pct}_{c,y} = \dfrac{\text{athletes}_{c,y}}{\sum_{c} \text{athletes}_{c,y}} \times 100 $$

This is what drives the stacked bar chart showing how Olympic participation and regional balance have shifted across disrupted and undisrupted editions over time.

In [12]:
# ============================================================
# VIZ 1 — Athlete Participation by Continent & Country
# Disrupted Games: Olympic Participation Through Global Events
# ============================================================

def assign_continent(region):
    europe = {
        'Albania','Andorra','Austria','Belarus','Belgium','Bosnia and Herzegovina',
        'Bulgaria','Croatia','Cyprus','Czech Republic','Denmark','Estonia','Finland',
        'France','Germany','Greece','Hungary','Iceland','Ireland','Italy','Latvia',
        'Liechtenstein','Lithuania','Luxembourg','Malta','Moldova','Monaco',
        'Montenegro','Netherlands','Norway','Poland','Portugal','Romania','Russia',
        'San Marino','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland',
        'Ukraine','United Kingdom','Yugoslavia','Kosovo','North Macedonia',
        'UK','Armenia','Azerbaijan','Georgia','Macedonia'
    }
    americas = {
        'Argentina','Bolivia','Brazil','Canada','Chile','Colombia','Costa Rica',
        'Cuba','Dominican Republic','Ecuador','El Salvador','Guatemala','Haiti',
        'Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru',
        'Puerto Rico','Trinidad','Uruguay','USA','Venezuela','Bahamas','Barbados',
        'Belize','Guyana','Suriname','Trinidad and Tobago',
        'Virgin Islands, US','Virgin Islands, British','American Samoa','Bermuda',
        'Cayman Islands','Aruba','Curacao','Antigua','Grenada','Saint Kitts',
        'Saint Vincent','Saint Lucia','Dominica','Guam','Boliva'
    }
    asia = {
        'Afghanistan','Bahrain','Bangladesh','Bhutan','Brunei','Cambodia','China',
        'India','Indonesia','Iran','Iraq','Israel','Japan','Jordan','Kazakhstan',
        'Kuwait','Kyrgyzstan','Laos','Lebanon','Malaysia','Maldives','Mongolia',
        'Myanmar','Nepal','North Korea','Oman','Pakistan','Palestine','Philippines',
        'Qatar','Saudi Arabia','Singapore','South Korea','Sri Lanka','Syria',
        'Taiwan','Tajikistan','Thailand','Timor-Leste','Turkey','Turkmenistan',
        'UAE','Uzbekistan','Vietnam','Yemen','United Arab Emirates'
    }
    africa = {
        'Algeria','Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon',
        'Cape Verde','Central African Republic','Chad','Comoros','Congo',
        'Djibouti','Egypt','Eritrea','Ethiopia','Gabon','Gambia','Ghana','Guinea',
        'Guinea-Bissau','Ivory Coast','Kenya','Lesotho','Liberia','Libya',
        'Madagascar','Malawi','Mali','Mauritania','Mauritius','Morocco',
        'Mozambique','Namibia','Niger','Nigeria','Rwanda','Senegal',
        'Sierra Leone','Somalia','South Africa','South Sudan','Sudan',
        'Swaziland','Tanzania','Togo','Tunisia','Uganda','Zambia','Zimbabwe',
        'Republic of Congo','Democratic Republic of the Congo','Equatorial Guinea',
        'Seychelles','Sao Tome and Principe'
    }
    oceania = {
        'Australia','Fiji','Kiribati','Marshall Islands','Micronesia','Nauru',
        'New Zealand','Palau','Papua New Guinea','Samoa','Solomon Islands',
        'Tonga','Tuvalu','Vanuatu','Cook Islands'
    }
    if pd.isna(region): return 'Unknown'
    if region in europe: return 'Europe'
    if region in americas: return 'Americas'
    if region in asia: return 'Asia'
    if region in africa: return 'Africa'
    if region in oceania: return 'Oceania'
    if region == 'Individual Olympic Athletes': return 'Unknown'
    return 'Unknown'

# --- Load & filter to Summer only (Viz 1 is Summer-only per spec) ---
summer = athletes[athletes['Season'] == 'Summer'].copy()
if 'region' not in summer.columns:
    summer = summer.merge(noc[['NOC', 'region']], on='NOC', how='left')
summer['Continent'] = summer['region'].apply(assign_continent)

print("Continent value counts:")
print(summer['Continent'].value_counts())
print("\nUnresolved regions:", summer[summer['Continent']=='Unknown']['region'].unique())

# --- VIZ 1a: Stacked bar data — unique athletes per Year x Continent ---
viz1_continent = (summer[summer['Continent'] != 'Unknown']
    .groupby(['Year', 'Continent'])['ID'].nunique()
    .reset_index(name='athletes'))

totals = viz1_continent.groupby('Year')['athletes'].transform('sum')
viz1_continent['pct'] = (viz1_continent['athletes'] / totals * 100).round(2)
viz1_continent = viz1_continent.sort_values(['Year', 'Continent']).reset_index(drop=True)

print(f"\n✅ viz1_continent — {len(viz1_continent)} rows")
print(viz1_continent.head(10))

# --- VIZ 1b: Line chart data — unique athletes per Year x NOC (country detail mode) ---
viz1_country = (summer[summer['Continent'] != 'Unknown']
    .groupby(['Year', 'NOC', 'region', 'Continent'])['ID'].nunique()
    .reset_index(name='athletes')
    .rename(columns={'region': 'Country'}))

viz1_country = viz1_country.sort_values(['Continent', 'Country', 'Year']).reset_index(drop=True)

print(f"\n✅ viz1_country — {len(viz1_country)} rows")
print(viz1_country.head(8))

# --- Country dropdown lookup: NOCs per continent, sorted by descending total historical participation ---
noc_totals = (summer[summer['Continent'] != 'Unknown']
    .groupby(['NOC', 'region', 'Continent'])['ID'].nunique()
    .reset_index(name='total_athletes')
    .rename(columns={'region': 'Country'}))

noc_totals = noc_totals.sort_values(['Continent', 'total_athletes'], ascending=[True, False]).reset_index(drop=True)

country_dropdown_lookup = {
    cont: grp[['NOC', 'Country', 'total_athletes']].to_dict('records')
    for cont, grp in noc_totals.groupby('Continent')
}

print(f"\n✅ country_dropdown_lookup — {len(country_dropdown_lookup)} continents")
for cont in country_dropdown_lookup:
    print(cont, '- top 3:', country_dropdown_lookup[cont][:3])

# --- Export to JSON ---
import json

viz1_continent.to_json(os.path.join(DATA_OUT, 'viz1_continent_athletes.json'), orient='records')
print(f"\n✅ viz1_continent_athletes.json saved — {len(viz1_continent)} rows")

viz1_country.to_json(os.path.join(DATA_OUT, 'viz1_country_athletes.json'), orient='records')
print(f"✅ viz1_country_athletes.json saved — {len(viz1_country)} rows")

with open(os.path.join(DATA_OUT, 'viz1_country_lookup.json'), 'w') as f:
    json.dump(country_dropdown_lookup, f)
print(f"✅ viz1_country_lookup.json saved — {len(country_dropdown_lookup)} continents")


Continent value counts:
Continent
Europe      127643
Americas     44062
Asia         28564
Africa       11865
Oceania       9955
Unknown        463
Name: count, dtype: int64

Unresolved regions: [nan 'Individual Olympic Athletes']

✅ viz1_continent — 140 rows
   Year Continent  athletes    pct
0  1896  Americas        14   7.95
1  1896    Europe       161  91.48
2  1896   Oceania         1   0.57
3  1900  Americas        90   7.35
4  1900      Asia         2   0.16
5  1900    Europe      1129  92.24
6  1900   Oceania         3   0.25
7  1904    Africa         8   1.23
8  1904  Americas       584  89.85
9  1904    Europe        56   8.62



✅ viz1_country — 2785 rows
   Year  NOC  Country Continent  athletes
0  1964  ALG  Algeria    Africa         1
1  1968  ALG  Algeria    Africa         3
2  1972  ALG  Algeria    Africa         5
3  1980  ALG  Algeria    Africa        54
4  1984  ALG  Algeria    Africa        33
5  1988  ALG  Algeria    Africa        42
6  1992  ALG  Algeria    Africa        35
7  1996  ALG  Algeria    Africa        45

✅ country_dropdown_lookup — 5 continents
Africa - top 3: [{'NOC': 'EGY', 'Country': 'Egypt', 'total_athletes': 999}, {'NOC': 'RSA', 'Country': 'South Africa', 'total_athletes': 952}, {'NOC': 'NGR', 'Country': 'Nigeria', 'total_athletes': 565}]
Americas - top 3: [{'NOC': 'USA', 'Country': 'USA', 'total_athletes': 7970}, {'NOC': 'CAN', 'Country': 'Canada', 'total_athletes': 3543}, {'NOC': 'BRA', 'Country': 'Brazil', 'total_athletes': 2024}]
Asia - top 3: [{'NOC': 'JPN', 'Country': 'Japan', 'total_athletes': 3273}, {'NOC': 'CHN', 'Country': 'China', 'total_athletes': 2353}, {'NOC': 'KOR', 

**Viz 2 — Medal Efficiency by Country per Decade.** Raw medal counts favor big countries that just send more athletes, so we also compute an efficiency ratio that levels the playing field:

\( \text{efficiency} = \dfrac{\text{medals won}}{\text{athletes sent}} \)

We also track each country's women's participation rate the same way, flag decades with fewer than 5 athletes as `sparse` (too little data to trust), and find each country's best-performing sport per decade and its Gold/Silver/Bronze breakdown.

In [13]:
"""Export browser-ready, Summer-only aggregates for OlympicLens Viz 2."""
from pathlib import Path
import json
import pandas as pd

ROOT = (Path(__file__).resolve().parents[1] if "__file__" in globals()
        else (Path.cwd().parent / "template").resolve())
SOURCE = ROOT.parent / "python" / "olympics_master_cleaned.csv"
OUTPUT = ROOT / "public" / "data" / "viz2_efficiency_yearly.json"

# Geometry names in public/data/countries.geo.json. Historical identities remain
# separate records; geometry_name is only the modern outline used to draw them.
NAME_FIX = {
    "UK": "United Kingdom", "USA": "United States of America",
    "Tanzania": "United Republic of Tanzania", "Bahamas": "The Bahamas",
    "Guinea-Bissau": "Guinea Bissau", "Congo": "Republic of the Congo",
    "Republic of Congo": "Republic of the Congo", "Trinidad": "Trinidad and Tobago",
    "Timor-Leste": "East Timor", "Serbia": "Republic of Serbia",
    "Boliva": "Bolivia", "": "",
}
HISTORICAL = {
    "URS": ("Russia", "Russia"), "EUN": ("Russia", "Russia"),
    "GDR": ("Germany", "Germany"), "FRG": ("Germany", "Germany"),
    "YUG": ("Republic of Serbia", "Serbia"), "SCG": ("Republic of Serbia", "Serbia"),
    "ANZ": ("Australia", "Australia"), "BOH": ("Czech Republic", "Czech Republic"),
    "TCH": ("Czech Republic", "Czech Republic"), "SAA": ("Germany", "Germany"),
}
HISTORICAL_LABELS = {
    "URS": "Soviet Union", "EUN": "Unified Team", "GDR": "East Germany",
    "FRG": "West Germany", "YUG": "Yugoslavia", "SCG": "Serbia and Montenegro",
    "ANZ": "Australasia", "BOH": "Bohemia", "TCH": "Czechoslovakia",
    "SAA": "United Team of Germany",
}
UNSUPPORTED = {"IOA", "ROT", "UNK", "MIX"}


def clean_event_list(frame):
    if frame.empty:
        return []
    return sorted(frame["Event"].dropna().unique().tolist())


def main():
    geometry_names = {f["properties"]["name"] for f in json.loads(
        (ROOT / "public" / "data" / "countries.geo.json").read_text(encoding="utf-8")
    )["features"]}
    use = ["ID", "NOC", "region", "Continent", "Year", "Season", "City", "Sport", "Event", "Medal"]
    df = pd.read_csv(SOURCE, usecols=use, na_values=[], keep_default_na=False)
    df = df[(df.Season == "Summer") & (df.Year.isin([
        1896,1900,1904,1908,1912,1920,1924,1928,1932,1936,1948,1952,1956,
        1960,1964,1968,1972,1976,1980,1984,1988,1992,1996,2000,2004,2008,2012,2016
    ]))].copy()
    df["ID"] = pd.to_numeric(df.ID)
    medal_rows = df[df.Medal.isin(["Gold", "Silver", "Bronze"])].copy()
    medals = medal_rows.drop_duplicates(["Year", "NOC", "Sport", "Event", "Medal"])

    # First facts are scoped nationally and by sport. Ties within an edition retain every event.
    facts = {}
    for noc, noc_df in medal_rows.groupby("NOC"):
        for scope, scoped in [("All sports", noc_df)] + list(noc_df.groupby("Sport")):
            gold = scoped[scoped.Medal == "Gold"]
            fm_year = int(scoped.Year.min()) if not scoped.empty else None
            fg_year = int(gold.Year.min()) if not gold.empty else None
            facts[(noc, scope)] = {
                "first_medal_year": fm_year,
                "first_medal_events": clean_event_list(scoped[scoped.Year == fm_year]) if fm_year else [],
                "first_gold_year": fg_year,
                "first_gold_events": clean_event_list(gold[gold.Year == fg_year]) if fg_year else [],
            }

    records = []
    for (year, noc), edition in df.groupby(["Year", "NOC"]):
        scopes = [("All sports", edition)] + list(edition.groupby("Sport"))
        for scope, participants in scopes:
            won = medals[(medals.Year == year) & (medals.NOC == noc)]
            if scope != "All sports":
                won = won[won.Sport == scope]
            counts = won.Medal.value_counts()
            breakdown = {m: int(counts.get(m, 0)) for m in ["Gold", "Silver", "Bronze"]}
            total = sum(breakdown.values())
            sport_counts = won.groupby("Sport").size().sort_values(ascending=False)
            region = HISTORICAL_LABELS.get(noc, str(participants.region.iloc[0]))
            geom = NAME_FIX.get(region, region)
            mapped_to = None
            if noc in HISTORICAL:
                geom, mapped_to = HISTORICAL[noc]
            mapping_status = ("historical" if mapped_to else
                              ("direct" if geom in geometry_names else "unsupported"))
            fact = facts.get((noc, scope), {})
            records.append({
                "year": int(year), "noc": noc, "label": region, "sport": scope,
                "continent": str(participants.Continent.iloc[0]), "host": str(participants.City.iloc[0]),
                "geometry_name": geom, "mapping_status": mapping_status, "mapped_to": mapped_to,
                "athletes": int(participants.ID.nunique()), **breakdown, "medals": total,
                "efficiency": round(total / participants.ID.nunique(), 5),
                "best_sport": str(sport_counts.index[0]) if len(sport_counts) else None,
                **fact,
            })

    payload = {
        "meta": {
            "season": "Summer", "years": sorted(df.Year.unique().astype(int).tolist()),
            "medal_dedup_key": ["Year", "NOC", "Sport", "Event", "Medal"],
            "athlete_key": "ID", "unsupported_nocs": sorted({r["noc"] for r in records if r["mapping_status"] == "unsupported"}),
            "historical_geometry_map": {k: v[0] for k, v in HISTORICAL.items()},
        },
        "records": records,
    }
    OUTPUT.write_text(json.dumps(payload, separators=(",", ":")), encoding="utf-8")
    print(f"Wrote {len(records):,} records for {len(payload['meta']['years'])} Summer editions to {OUTPUT}")


if __name__ == "__main__":
    main()



Wrote 24,684 records for 28 Summer editions to C:\Users\namit\Desktop\2026-ss-st192675\assignment_05\template\public\data\viz2_efficiency_yearly.json


**Viz 3 — Country–Sport Dominance Network.** For each of our three eras, we build a country–sport edge whenever a country has won at least 5 medals in that sport — this threshold keeps the network graph readable by cutting out one-off, statistically insignificant results and leaving only genuine dominance patterns.

In [14]:
"""Export browser-ready sport programme hierarchy for OlympicLens Viz 3."""
from pathlib import Path
import json
import pandas as pd

SOURCE = Path('olympics_master_cleaned.csv')
MAPPING_SOURCE = Path('../template/public/data/viz3_sport_families.json')
OUTPUT = Path('viz3_sport_tree.json')

CANCELLED_YEARS = [1916, 1940, 1944]
SUMMER_YEARS = list(range(1896, 2017, 4))
SUMMER_YEARS = [y for y in SUMMER_YEARS if y not in CANCELLED_YEARS]


def load_family_mapping():
    """Reuse the dropped Viz 3 mapping where available; unknown sports go to Other."""
    if MAPPING_SOURCE.exists():
        payload = json.loads(MAPPING_SOURCE.read_text(encoding="utf-8"))
        mapping = payload.get("mapping", {})
        families = [f for f in payload.get("families", []) if f != "All"]
        return mapping, families
    return {}, ["Athletics", "Aquatics", "Combat", "Ball Sports",
                "Gymnastics", "Cycling/Wheels", "Other"]


def gaps_for(active_years, years):
    observed = set(active_years)
    if not active_years:
        return []
    present = [y for y in years if y in observed]
    gaps = []
    for start, end in zip(present, present[1:]):
        missing = [y for y in years if start < y < end and y not in observed]
        if missing:
            gaps.append([missing[0], missing[-1]])
    return gaps


def main():
    mapping, families = load_family_mapping()
    use = ["NOC", "Year", "Season", "Sport", "Sex"]
    df = pd.read_csv(SOURCE, usecols=use, na_values=[], keep_default_na=False)
    df = df[(df.Season == "Summer") & df.Year.isin(SUMMER_YEARS)].copy()
    df["Year"] = pd.to_numeric(df["Year"]).astype(int)
    years = sorted(df.Year.unique().tolist())

    sport_year = {}
    for sport, sport_df in df.groupby("Sport"):
        if sport_df.empty:
            continue
        sport_year[sport] = {}
        for year, edition in sport_df.groupby("Year"):
            women_pct = float((edition.Sex == "F").mean()) if len(edition) else 0.0
            sport_year[sport][str(int(year))] = {
                "active": True,
                "nocCount": int(edition.NOC.nunique()),
                "womenPct": round(women_pct, 6),
            }

    children = {family: [] for family in families}
    if "Other" not in children:
        children["Other"] = []
        families.append("Other")

    for sport in sorted(sport_year):
        active_years = sorted(int(y) for y in sport_year[sport])
        first_year = active_years[0]
        last_active = active_years[-1]
        last_year = None if last_active == max(years) else last_active
        gaps = gaps_for(active_years, years)
        status = "reinstated" if gaps else ("removed" if last_year is not None else "active")
        family = mapping.get(sport, "Other")
        if family not in children:
            family = "Other"
        children[family].append({
            "name": sport, "firstYear": first_year, "lastYear": last_year,
            "status": status, "gaps": gaps, "yearlyStats": sport_year[sport],
        })

    tree = {"name": "Olympic Programme", "children": [
        {"name": family, "children": children[family]}
        for family in families if children[family]
    ]}
    payload = {"meta": {"cancelledYears": CANCELLED_YEARS,
                          "years": years, "families": [n["name"] for n in tree["children"]]},
               "tree": tree}
    OUTPUT.write_text(json.dumps(payload, separators=(",", ":")), encoding="utf-8")
    sports = [sport for family in tree["children"] for sport in family["children"]]
    print(f"Wrote {len(sports)} sports across {len(tree['children'])} families to {OUTPUT}")
    print(f"Sports with gaps: {sum(bool(s['gaps']) for s in sports)}")
    print(json.dumps({"name": tree["name"], "children": [
        {"name": f["name"], "children": f["children"][:2]}
        for f in tree["children"][:1]
    ]}, indent=2))
    print(f"Cancelled years in meta: {payload['meta']['cancelledYears']}")
    print(f"Cancelled years absent from yearlyStats: {all(str(y) not in s['yearlyStats'] for s in sports for y in CANCELLED_YEARS)}")


main()


Wrote 52 sports across 7 families to viz3_sport_tree.json
Sports with gaps: 16
{
  "name": "Olympic Programme",
  "children": [
    {
      "name": "Aquatics",
      "children": [
        {
          "name": "Diving",
          "firstYear": 1904,
          "lastYear": null,
          "status": "active",
          "gaps": [],
          "yearlyStats": {
            "1904": {
              "active": true,
              "nocCount": 2,
              "womenPct": 0.0
            },
            "1908": {
              "active": true,
              "nocCount": 9,
              "womenPct": 0.0
            },
            "1912": {
              "active": true,
              "nocCount": 10,
              "womenPct": 0.162791
            },
            "1920": {
              "active": true,
              "nocCount": 14,
              "womenPct": 0.271429
            },
            "1924": {
              "active": true,
              "nocCount": 14,
              "womenPct": 0.311111
            }

## 9. Export Olympic Disruptions (Boycotts, Cancellations, Wars, Terrorism)

Here's the piece that doesn't come from either dataset above: boycotts, cancelled Games, and Olympic-specific shocks like the 1972 Munich attack. None of this is something we can compute from athlete records or general world-events data — it has to be sourced and entered by hand.

So this table is **manually curated**, cross-checked against Wikipedia, Britannica, Olympedia, and US State Department archives, and kept as its own small file rather than merged into `world_events_clean.csv`. Keeping it separate is deliberate: it makes clear in the notebook exactly which parts of our story are computed from data and which parts are researched historical annotation, which matters for the project's data provenance.

Make sure `olympic_disruptions.csv` is sitting in `python/` (same folder as this notebook) before running the cell below.

In [15]:
disruptions_path = os.path.join(LAB, 'olympic_disruptions.csv')
disruptions = pd.read_csv(r"C:\Users\namit\Desktop\2026-ss-st192675\assignment_05\python\olympic_disruptions.csv")

def assign_category(row):
    if row['Type'] in ['World War', 'Cancelled']:
        return 'Wars/Cancellations'
    if row['Type'] == 'Boycott':
        return 'Boycotts'
    if row['Type'] in ['Terrorism', 'Security']:
        return 'Security Events'
    if row['Type'] in ['Natural Disaster', 'Public Health']:
        return 'Geographical Calamity'
    return 'Political Events'

disruptions['Category'] = disruptions.apply(assign_category, axis=1)

disruptions = disruptions[[
    'Year', 'Games', 'Type', 'Category', 'Impact_Level', 'Label', 'Host_City', 'Host_Country',
    'Countries_Involved', 'Reason', 'Story', 'Display_Style'
]]

disruptions.to_json(os.path.join(DATA_OUT, 'olympic_disruptions.json'), orient='records')
print(f'✅ olympic_disruptions.json saved — {len(disruptions)} rows')

✅ olympic_disruptions.json saved — 34 rows


## 10. Summary

One last sanity check — this loops through every file we expect to have produced and confirms it actually exists and isn't empty. If anything shows up as `MISSING`, that's the first thing to fix before touching the frontend.

In [16]:
# --- Output file check ---
files = [
    (PYTHON_OUT, 'olympics_master_cleaned.csv'),
    (PYTHON_OUT, 'world_events_clean.csv'),
    (DATA_OUT, 'viz1_continent_athletes.json'),
    (DATA_OUT, 'viz1_country_athletes.json'),
    (DATA_OUT, 'viz1_country_lookup.json'),
    (DATA_OUT, 'viz2_efficiency_yearly.json'),
    (DATA_OUT, 'viz3_network.json'),
    (DATA_OUT, 'olympic_disruptions.json'),
]
print('Output file check:')
for folder, fname in files:
    path = os.path.join(folder, fname)
    size = os.path.getsize(path) / 1024 if os.path.exists(path) else None
    status = f'{size:.1f} KB' if size else 'Nope MISSING'
    print(f'  {"Done" if size else "Nope"} {fname:<40} {status}')

Output file check:
  Done olympics_master_cleaned.csv              45629.0 KB
  Done world_events_clean.csv                   39.7 KB
  Done viz1_continent_athletes.json             8.4 KB
  Done viz1_country_athletes.json               222.3 KB
  Done viz1_country_lookup.json                 13.6 KB
  Done viz2_efficiency_yearly.json              9432.2 KB
  Done viz3_network.json                        134.0 KB
  Done olympic_disruptions.json                 15.2 KB
